In [1]:
import ray
ray.shutdown()

In [38]:
# Import all required packages
import pandas as pd                   # For data manipulation and analysis
import io
import gymnasium as gym
import numpy as np
import os
import pickle
import copy
import random
import time
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import matplotlib.ticker as ticker
import warnings
from concurrent.futures import ProcessPoolExecutor
import multiprocessing
import torch
import networkx as nx
from tqdm import tqdm
from scipy.stats import ttest_ind
import seaborn as sns

# Import custom zero counting function
def count_zeros(arr):
    return np.count_nonzero(arr < 0.00001)

# Import custom environment .py file
import env_TA as ce 

#User Determined Settings
graphName = "G10x10"  # Graph nodes and edges to use
env_params = {'deterministic_agent': False,
              'multiple_interdiction_attempts': True,
              'attacker_strategy': 'zero_sum',  # canalize   isolate   divert  zero_sum
              'training_budget_range': (5, 15),  #G5x5: zero_sum/isolate: (5,15), canalize/divert: (10,20) G10x10: zero_sum/isolate: (15,30), canalize/divert: (20,40)   #UKR: zero_sum/isolate: (10,20), canalize/divert: (15,25)
              'max_path_length': 6,  #G5x5: 6,  G10x10: 13, UKR: 16
              'sample_size': None #None #None #000,
             }
version_name = "v01_01_opt"

## Create Environment
# Create nodes and edges
node_filename = f"{graphName}_Nodes.csv"  # Dynamically include graphName
edge_filename = f"{graphName}_Edges.csv"  # Dynamically include graphName

nodes, edges = ce.create_nodes_edges(node_filename, edge_filename)

# Load Environment
env = ce.CustomEnv(nodes, edges, **env_params)

# Get the current working directory
current_dir = os.getcwd()

## Load Optimal Solutions
if env_params['multiple_interdiction_attempts'] == True:
    MI_letter = 'M'
else:
    MI_letter = 'B'
    
save_filename = f"{graphName}_{env_params['attacker_strategy']}_{MI_letter}_solution_{version_name}.pkl"  

save_path = os.path.join(current_dir, '..', 'Solutions', save_filename)

# Open the pickle file in binary read mode and load the data
with open(save_path, "rb") as f:
    results = pickle.load(f)

# Now you can access the saved data
print("Episodes completed:", results["last_episode"] + 1)
print("Mean Objective values:", np.mean(results["optimal_obj_vals"]))
#print("Interdiction sets:", results["all_optimal_interdiction_edges"])
print("Mean Solution times:", np.mean(results["optimal_solution_times"]))

optimal_obj_vals = results["optimal_obj_vals"]
all_optimal_interdiction_edges = results["all_optimal_interdiction_edges"]
optimal_solution_times = results["optimal_solution_times"]
all_states = results['states']



Episodes completed: 500
Mean Objective values: 204.638737142857
Mean Solution times: 738.6773238686686


In [ ]:
# Greedy Max-Flow Heuristic Solver
# Heuristic: At each step, solve for stochastic max flow, then interdict the valid edge with most expected (flow * probability of success)

num_of_scenarios = 500

# Pre-compute constants outside the loop
interdictable_edges_list = list(env.both_edges)

# Initialize arrays to store results
reference_objs = np.zeros(num_of_scenarios)
heuristic_best_rewards = np.zeros(num_of_scenarios)

# Initialize a dictionary to keep track of actions
action_tally = {i: 0 for i in env.both_edges}

all_heuristic_actions = []
heuristic_solution_times = np.zeros(num_of_scenarios)

for episode in tqdm(range(num_of_scenarios)):
    env_state = copy.deepcopy(all_states[episode][0])
    obs, _ = env.load_network_from_state(episode, env_state)

    # Determine Original Flow
    reference_objs[episode] = env.reference_obj
    if optimal_obj_vals[episode] == reference_objs[episode]:
        heuristic_best_rewards[episode] = np.nan
        heuristic_solution_times[episode] = np.nan
        all_heuristic_actions.append([])
        continue

    action_list = []
    action_list_append = action_list.append

    start_time = time.perf_counter()

    budget = obs['budget'][0]
    
    for _ in range(31):  # Max iterations (budget + safety margin)
        # Get valid action mask
        action_mask = env.mask_fn(depth=1)
        valid_actions = np.where(action_mask[:env.num_interdictable] == 1)[0]
        
        # Check if no valid actions remain
        if len(valid_actions) == 0:
            # Select "do nothing" action
            action = 500
            obs, reward, done, _, _ = env.step(action)
            break
        
        # Solve for stochastic max flow to get expected edge flows
        if env.deterministic_outcomes:
            # Deterministic case: solve max flow directly
            _, flows = env.solve_max_flow(routing_assumption=env.attacker_strategy)
        else:
            # Stochastic case: get expected flows from stochastic calculation
            _, flows = env._calculate_stochastic_objective_and_flow(
                strategy_type=env.attacker_strategy, 
                return_full_flows=True
            )
        
        # Calculate expected flow on each valid edge (sum of forward and reverse flow)
        edge_flows = np.zeros(env.num_interdictable)
        
        # Get interdiction probabilities for all interdictable edges
        interdiction_probs = env.state['edge_interdiction_probability'][:env.num_interdictable]
        interdiction_attempts = env.state['edge_interdicted'][:env.num_interdictable]
        costs = env.state['edge_costs'][:env.num_interdictable]
        
        for idx in valid_actions:
            edge = interdictable_edges_list[idx]
            reverse_edge = (edge[1], edge[0])
            forward_flow = flows.get(edge, 0)
            reverse_flow = flows.get(reverse_edge, 0)
            edge_flows[idx] = forward_flow + reverse_flow
        
        # Select the valid action with maximum (expected flow * interdiction probability)
        # Only consider valid actions for selection

        if env.attacker_strategy == "zero_sum":
            heuristic_values = edge_flows # * (1-(1-interdiction_probs)**(interdiction_attempts+1)) #/ costs
        elif env.attacker_strategy == "isolate":
            pass
        elif env.attacker_strategy == "canalize":
            pass
        elif env.attacker_strategy == "divert":
            pass
        
        masked_values = np.full(env.num_interdictable, -np.inf)
        masked_values[valid_actions] = heuristic_values[valid_actions]
        action = int(np.argmax(masked_values))
        
        # Step the environment with selected action
        obs, reward, done, _, _ = env.step(action)

        # Tally actions
        if action != 500:
            action_key = interdictable_edges_list[action]
            action_tally[action_key] += 1
            action_list_append(action_key)
            
        if done:
            break
            
    end_time = time.perf_counter()
    heuristic_solution_times[episode] = (end_time - start_time)
    all_heuristic_actions.append(sorted(action_list))

    if all_heuristic_actions[episode] == all_optimal_interdiction_edges[episode]:
        objVal = optimal_obj_vals[episode]
    else:
        # Get final objective value
        if env_params['attacker_strategy'] == "zero_sum":
            objVal, _ = env._compute_objective_and_flows()
        elif env_params['attacker_strategy'] == "isolate":
            objVal, _ = env._calculate_isolate_objective_and_flows()
        elif env_params['attacker_strategy'] == "canalize":
            objVal, _ = env._calculate_canalize_objective_and_flows()
        elif env_params['attacker_strategy'] == "divert":
            objVal, _ = env._calculate_divert_objective_and_flows()

    heuristic_best_rewards[episode] = objVal

# Print Results
print("Optimal Time to Solve (Mean):", np.mean(optimal_solution_times[0:num_of_scenarios]))
print("")

if env_params['attacker_strategy'] == "zero_sum":
    relative_errors = np.clip(heuristic_best_rewards - optimal_obj_vals[0:num_of_scenarios], 0, None) / (reference_objs - optimal_obj_vals[0:num_of_scenarios])

elif env_params['attacker_strategy'] == "canalize":
    relative_errors = np.clip(optimal_obj_vals[0:num_of_scenarios] - heuristic_best_rewards, 0, None) / (optimal_obj_vals[0:num_of_scenarios])

elif env_params['attacker_strategy'] == "isolate":
    relative_errors = np.clip(((reference_objs[0:num_of_scenarios] - optimal_obj_vals[0:num_of_scenarios]) - 
                               (reference_objs[0:num_of_scenarios] - heuristic_best_rewards)) / (reference_objs[0:num_of_scenarios] - optimal_obj_vals[0:num_of_scenarios]), 0, None)
elif env_params['attacker_strategy'] == "divert":
    relative_errors = (optimal_obj_vals[0:num_of_scenarios] - heuristic_best_rewards) / optimal_obj_vals[0:num_of_scenarios]

# Calculate mean relative error
mean_relative_error = np.nanmean(relative_errors)
print("Greedy Max-Flow * Probability / Cost Heuristic Results:")
print("  Optimal solutions found:", count_zeros(relative_errors))
print("  Mean Relative Error:", mean_relative_error)
print("  Heuristic Time to Solve (Mean):", np.nanmean(heuristic_solution_times))
print("")

  1%|          | 5/500 [00:05<10:49,  1.31s/it]

In [24]:
start = 51
end = start+1
#end = 30
print('Reference Objs: ', reference_objs[start:end])
print('Optimal Objs: ',optimal_obj_vals[start:end])
print('Heuristic Objs: ',heuristic_best_rewards[start:end])
print('Relative Errors: ',relative_errors[start:end])
print('Optimal Actions: ',all_optimal_interdiction_edges[start:end])
print('Heuristic Actions: ',all_heuristic_actions[start:end])

Reference Objs:  [168.]
Optimal Objs:  [69.]
Heuristic Objs:  [138.65317499]
Relative Errors:  [0.57819438]
Optimal Actions:  [[(17, 22), (18, 23), (19, 24), (21, 26)]]
Heuristic Actions:  [[(7, 23), (19, 54), (30, 31), (37, 38), (40, 41)]]


In [6]:
np.where(relative_errors[0:500]>1)

(array([51]),)

In [8]:
relative_errors

array([0.33769251, 0.10869757, 1.        , 0.05445757, 0.        ,
       0.        , 0.        , 0.00431032, 0.        , 0.        ,
       0.9726234 , 1.        , 0.48934427, 0.28775862, 0.4842534 ,
       0.34823645, 0.23884294, 0.42581138, 0.44698138, 0.10790144,
       0.23086157, 0.36286408, 0.32137403, 0.35444797, 0.70503703,
       0.        , 0.35385483, 0.09336952, 0.09186852, 0.49126992,
       0.11704475, 0.06012281, 1.        , 0.29029301, 0.14373087,
       0.11851596, 0.14853466, 0.14223271, 0.        , 0.        ,
       0.94872093, 0.96393939, 0.32173616, 0.2881356 , 1.        ,
       0.6363679 , 0.23165331, 0.1637168 , 0.10176988, 0.38435342,
       0.45676403, 1.        , 0.99481117, 0.        , 0.09186363,
       0.76881066, 0.25625053, 0.12011176, 0.48143623, 0.        ,
       0.80507657, 0.33876865, 0.24647365, 0.25462326, 0.08411588,
       1.        , 0.        , 0.13041858, 1.        , 0.11680721,
       0.10971897, 0.42186236, 0.34731932, 0.2237678 , 0.44384